# Figure 6: quadratic-mode coupling coefficients

Input: coupling tables distributed with `tgr`. The tables store the complex quadratic-mode amplitude ratios as functions of remnant spin. Both axes have units of $1/M_f$.

In [ ]:
from importlib.resources import files
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

fig_width = 3 * 246.0 / 72.27
fig_height = fig_width * (np.sqrt(5) - 1.0) / 2.0
mpl.rcParams.update({
    'axes.labelsize': 24,
    'font.family': 'serif',
    'font.serif': 'Computer Modern Roman',
    'font.size': 24,
    'legend.fontsize': 20,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'axes.grid': True,
    'text.usetex': True,
    'savefig.dpi': 100,
    'lines.markersize': 14,
    'figure.figsize': [fig_width, fig_height],
})

TABLE_DIR = files('tgr').joinpath('data')
quadratic_modes = [
    '220220', '220221', '221221', '220222', '221222',
    '222222', '220223', '221223', '222223', '223223',
]
markers = ['o', '^', 'D', 's', 'p', 'X', 'v', '<', '>', 'h']

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(1, 2, sharey=True)
fig.subplots_adjust(wspace=0.04)
for index, mode in enumerate(quadratic_modes):
    table = np.load(TABLE_DIR / f'spin_{mode}_interpolation.npy', allow_pickle=True).item()
    select = table['spin'] > 0.55
    spin = table['spin'][select]
    ratio = table[f'{mode}_complex_ratio'][select]
    order = np.argsort(spin)
    spin = spin[order]
    ratio = ratio[order]
    for axis in (ax_left, ax_right):
        axis.plot(ratio.real, ratio.imag, color='gray', alpha=0.5, linewidth=1)
        scatter = axis.scatter(
            ratio.real, ratio.imag, c=spin, marker=markers[index], cmap='viridis',
            s=150, edgecolors='black', linewidth=0.8,
            vmin=spin.min(), vmax=spin.max(), label=f'${mode[:3]}\\times{mode[3:]}$')

ax_left.set_xlim(None, 0.205)
ax_right.set_xlim(0.230, None)
ax_left.spines['right'].set_visible(False)
ax_right.spines['left'].set_visible(False)
ax_left.yaxis.tick_left()
ax_right.yaxis.tick_right()

d = 0.008
kwargs = dict(transform=ax_left.transAxes, color='k', clip_on=False, linewidth=1)
ax_left.plot((1-d, 1+d), (-d, +d), **kwargs)
ax_left.plot((1-d, 1+d), (1-d, 1+d), **kwargs)
kwargs.update(transform=ax_right.transAxes)
ax_right.plot((-d, +d), (1-d, 1+d), **kwargs)
ax_right.plot((-d, +d), (-d, +d), **kwargs)

colorbar = plt.colorbar(scatter, ax=[ax_left, ax_right], shrink=0.8)
colorbar.set_label('Remnant Spin')
fig.text(0.43, 0.00, 'Real Part [$1/M_\\mathrm{f}$]', ha='center', va='bottom')
ax_left.set_ylabel('Imaginary Part [$1/M_\\mathrm{f}$]')
ax_left.legend(ncols=2, fontsize=11, title='Modes', title_fontsize=18)
for axis in (ax_left, ax_right):
    axis.grid(True, alpha=0.4, linewidth=0.5)

output = ROOT / 'figures/theory.pdf'
fig.savefig(output, bbox_inches='tight')
plt.show()